# Environment Setup

In [1]:
import os

# Hadoop (must be set BEFORE Spark starts)
os.environ["HADOOP_HOME"] = r"C:\hadoop-3.0.0"
os.environ["hadoop.home.dir"] = r"C:\hadoop-3.0.0"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

In [2]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["SPARK_HOME"] = r"C:\Users\HP\Downloads\spark-3.5.8-bin-hadoop3\spark-3.5.8-bin-hadoop3"
os.environ["PATH"] = os.environ["SPARK_HOME"] + r"\bin;" + os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler, StringIndexer
)
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier, GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

import time
import numpy as np

# Initialize Spark Session

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Reddit_Finance_Distributed_ML_Pipeline") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.python.worker.reuse", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .getOrCreate()

# PySpark Data Engineering

## Data Ingestion and Storage Design

In [5]:
DATA_PATH = r"E:\Datasets\00_combined.csv"

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(DATA_PATH)

df_raw.printSchema()
df_raw.show(5, truncate=False)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- created_utc: integer (nullable = true)
 |-- created_datetime: timestamp (nullable = true)
 |-- score: integer (nullable = true)
 |-- num_comments: integer (nullable = true)
 |-- upvote_ratio: double (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- company: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+-------+------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Data Validation at Ingestion

In [6]:
from pyspark.sql import functions as F

# Mandatory columns check
required_cols = [
    "id", "title", "text", "created_utc", "created_datetime",
    "score", "num_comments", "upvote_ratio",
    "subreddit", "company", "year", "month"
]

missing_cols = set(required_cols) - set(df_raw.columns)
assert len(missing_cols) == 0, f"Missing columns: {missing_cols}"

# Drop invalid rows
df = df_raw.dropna(subset=["id", "company", "created_datetime"])

## Partitioning Strategy (Query-Aligned)

In [7]:
df = df.repartition("company")

# Distributed Data Processing Pipeline

## Basic Cleaning (Spark-native)

In [8]:
df_clean = (
    df
    .withColumn("title", F.lower(F.col("title")))
    .withColumn("text", F.lower(F.col("text")))
    .filter(F.length("title") > 10)
    .filter(F.length("text") > 30)
    .withColumn("score", F.col("score").cast("int"))
    .withColumn("num_comments", F.col("num_comments").cast("int"))
    .withColumn("upvote_ratio", F.col("upvote_ratio").cast("double"))
)

## Memory Management Strategy

In [9]:
df_clean.persist(StorageLevel.MEMORY_AND_DISK)
print("Cached rows:", df_clean.count())

Cached rows: 4906


In [10]:
df_clean.unpersist()

DataFrame[id: string, title: string, text: string, created_utc: int, created_datetime: timestamp, score: int, num_comments: int, upvote_ratio: double, subreddit: string, company: string, year: int, month: int]

## Broadcast Join Example

In [11]:
company_lookup = (
    df_clean.select("company")
    .distinct()
    .withColumn("company_flag", F.lit(1))
)

df_joined = df_clean.join(
    F.broadcast(company_lookup),
    on="company",
    how="left"
)

## Error Handling & Lineage

In [12]:
try:
    df_joined.count()
except Exception as e:
    print("Pipeline failure:", e)

# Performance Optimization

## Shuffle & Partition Tuning

In [13]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

In [14]:
spark.conf.set("spark.sql.shuffle.partitions", 150)